# Figure 3: Glycoproteome-wide association study for AD (TWAS)

## prepare data

In [ ]:
library(data.table)
library(tidyverse)
twas_res_df <- fread('data_output/twas/glyco_v202601.twas_res.no_ashinf.csv.gz')

# exclude sole-modification Trp C-mannosylation glycopeptiforms mislabeled as glycation
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')
coding_drop_gpids <- readLines('../../2_main_figures/Figure1/data/coding_variant_drop_gpids.txt')
stopifnot(length(coding_drop_gpids) == 21L)
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids, coding_drop_gpids)
w54_pattern <- paste0('_(', paste(excluded_gpids, collapse = '|'), ')$')
twas_res_df <- twas_res_df %>% filter(!str_detect(context, w54_pattern))

mr_res_df <- fread('data_output/mr/glyco_v202601.mr_res.csv.gz')

# exclude sole-modification Trp C-mannosylation glycopeptiforms mislabeled as glycation
mr_res_df <- mr_res_df %>% filter(!str_detect(context, w54_pattern))

mr_res_df_sig_adj <- mr_res_df %>% filter(str_detect(context, 'DLPFC_Klein_gpQTL_adjusted')) %>% .[
  meta_pval < 0.05 / .N &          # Bonferroni correction
  cpip >= 0.7 &                    # strong causal evidence
  num_CS >= 1 &                    # loose version (num_CS >= 1)
  Q_pval > 0.01 &                  # no heterogeneity
  I2 < 0.4                         # low heterogeneity
]

mr_res_df_sig_pqtl <- mr_res_df %>% filter(!str_detect(context, 'gpQTL')) %>% .[
  meta_pval < 0.05 / .N &          # Bonferroni correction
  cpip >= 0.7 &                    # strong causal evidence
  num_CS >= 1 &                    # loose version (num_CS >= 1)
  Q_pval > 0.01 &                  # no heterogeneity
  I2 < 0.4                         # low heterogeneity
]

### summarize significant TWAS

#### adjusted

In [ ]:
twas_adj <- twas_res_df %>% filter(type == "adjusted_gpQTL") 

n_imputable_genes <- twas_adj %>%
  filter(is_imputable) %>%                  # keep only imputable rows
  distinct(molecular_id, context, gwas_study) %>%         # count unique gene-context pairs
  nrow()
n_imputable_genes

# Compute cutoff using Bonferroni-style correction
adj_cutoff <- 0.05 / n_imputable_genes

### pQTL

In [ ]:
twas_pqtl <- twas_res_df %>% filter(type == "pQTL") 

n_imputable_genes <- twas_pqtl %>%
  filter(is_imputable) %>%                  # keep only imputable rows
  distinct(molecular_id, context, gwas_study) %>%         # count unique gene-context pairs
  nrow()
n_imputable_genes

# Compute cutoff using Bonferroni-style correction
pqtl_cutoff <- 0.05 / n_imputable_genes

# maf05 AD colocalization summary
ad_coloc_file <- "data_output/colocalization/maf05/glycoQTL_maf05/pQTL_glycoQTL_unadjusted_AD_summary_pv1e4.tsv"
stopifnot(file.exists(ad_coloc_file), length(unique(excluded_gpids)) == 77L)

gpQTL_colocalization <- data.table::fread(ad_coloc_file)
stopifnot("colocalized_outcomes" %in% names(gpQTL_colocalization))
stopifnot(!anyNA(gpQTL_colocalization$colocalized_outcomes))

outcome_tokens_ad <- str_split(gpQTL_colocalization$colocalized_outcomes, ";\\s*")
excl_gp_token_ad <- function(tok) {
  gp <- str_remove(tok, "^DLPFC_Klein_gpQTL_unadjusted_")
  str_detect(tok, "^DLPFC_Klein_gpQTL_unadjusted_gp_[0-9]+$") & gp %in% excluded_gpids
}
trimmed_tokens_ad <- lapply(outcome_tokens_ad, function(tok) tok[!excl_gp_token_ad(tok)])
n_trimmed_ad <- sum(lengths(outcome_tokens_ad)) - sum(lengths(trimmed_tokens_ad))
keep_row_ad <- lengths(trimmed_tokens_ad) >= 2L
cat(sprintf(
  "AD colocalization: trimmed %d excluded-gpID token(s); %d of %d rows dropped entirely (<2 members remain after trimming)
",
  n_trimmed_ad, sum(!keep_row_ad), nrow(gpQTL_colocalization)
))
gpQTL_colocalization <- gpQTL_colocalization[keep_row_ad, ]
gpQTL_colocalization[, colocalized_outcomes := vapply(
  trimmed_tokens_ad[keep_row_ad], paste, character(1), collapse = "; "
)]
stopifnot(!any(excl_gp_token_ad(unlist(str_split(
  gpQTL_colocalization$colocalized_outcomes, ";\\s*"
)))))

gpQTL_colocalization <- gpQTL_colocalization %>%
  filter(str_detect(colocalized_outcomes, fixed("AD_Bellenguez")))

# member-level AD colocalization. Panel a (adjusted glycoWAS): glycopeptide contexts from the adjusted colocalization run
# (same excluded-gpID trimming and >=2-member rule). Panel b (PWAS): genes whose pQTL itself is an AD CoS member.
ad_adj <- data.table::fread(sub("unadjusted_AD", "adjusted_AD", ad_coloc_file))
tok_adj <- str_split(ad_adj$colocalized_outcomes, ";\\s*")
tok_adj <- lapply(tok_adj, function(tok) tok[!(str_detect(tok, "^DLPFC_Klein_gpQTL_adjusted_gp_[0-9]+$") & str_extract(tok, "gp_[0-9]+$") %in% excluded_gpids)])
ad_adj <- ad_adj[lengths(tok_adj) >= 2L][, colocalized_outcomes := vapply(tok_adj[lengths(tok_adj) >= 2L], paste, character(1), collapse = "; ")]
ad_adj <- ad_adj[str_detect(colocalized_outcomes, fixed("AD_Bellenguez"))]
adj_tokens <- unique(unlist(strsplit(ad_adj$colocalized_outcomes, ";\\s*")))
colocboost_adj_contexts <- adj_tokens[str_detect(adj_tokens, "gpQTL_adjusted_gp_[0-9]+$")]
ad_tokens <- unique(unlist(strsplit(gpQTL_colocalization$colocalized_outcomes, ";\\s*")))
colocboost_pqtl_genes <- unique(str_extract(ad_tokens[str_detect(ad_tokens, "pQTL_ENSG[0-9]+$")], "ENSG[0-9]+$"))

gene_ref <- fread('resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list')

## Supplementary fig3 -- adjusted

In [ ]:
df <- twas_adj %>%
  mutate(sig = twas_pval < adj_cutoff) %>%
  group_by(molecular_id, context, gwas_study) %>%
  summarise(
    n_methods = n(),
    n_sig = sum(sig, na.rm = TRUE),
    best_pval = min(twas_pval, na.rm = TRUE),
    best_method_pval = min(twas_pval[is_selected_method], na.rm = TRUE),
    best_method_selected = any(sig & is_selected_method, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    pass = (n_sig / n_methods >= 0.5) | best_method_selected, # pass defined by two threshold, best or half methods satisfied
    final_pval = ifelse(n_sig / n_methods >= 0.5, best_pval, best_method_pval),
    # # fallback: if no selected method existed
    final_pval = ifelse(is.infinite(final_pval), best_pval, final_pval)
  ) %>%
  left_join(twas_adj, by = c("molecular_id", "context", "gwas_study")) %>%
  filter(twas_pval == final_pval) %>%
  distinct(context, .keep_all = TRUE) %>%
  mutate(
    logp = -log10(twas_pval) * sign(twas_z),
    pos = (start + end) / 2,
    is_coloc = context %in% colocboost_adj_contexts  # glycopeptiform level, adjusted coloc run
  ) %>%
  merge(gene_ref, by.x = 'molecular_id', by.y = 'gene_id') %>%
  mutate(peptide = paste0(gene_name, ": ", str_extract(context, 'gp_.+$')))


df$chr <- factor(df$chr, levels = as.character(1:22))

chr_info <- df %>%
  group_by(chr) %>%
  summarise(chr_len = max(pos)) %>%
  mutate(chr_start = lag(cumsum(chr_len), default = 0))

df <- df %>%
  left_join(chr_info, by = "chr") %>%
  mutate(cum_pos = pos + chr_start) %>%
  mutate(
    TWAS_colocaliztion = ifelse(
      is_coloc & pass, 
      'TWAS & Colocalization',
      ifelse(!is_coloc & pass, 'TWAS only', 'Not Significant')
    )
  )
twas_adj_sig_df = df
axis_df <- chr_info %>%
  mutate(center = chr_start + chr_len/2)

df <- mr_res_df_sig_adj %>% rename('molecular_id' = 'gene_name') %>% merge(df, . , all.x = T)

In [ ]:
library(ggplot2)
library(ggrepel)
library(dplyr)

# Match the main Figure 3a design
df <- df %>%
  mutate(
    MR = ifelse(!is.na(Q_pval), "Significant", "Not Significant"),
    chr_num = as.integer(as.character(chr)),
    point_group = case_when(
      is_coloc & pass ~ "Yes",   # colocalized points without GlycoWAS significance stay grey
      chr_num %% 2 == 1 ~ "No (odd)",
      TRUE ~ "No (even)"
    )
  ) %>% arrange(point_group == "Yes")

lab_df <- df %>%
  filter(pass) %>%
  group_by(gene_name) %>%
  slice_max(abs(logp), n = 1, with_ties = FALSE) %>%
  ungroup()

spp_f3a_adj <- ggplot(df, aes(x = cum_pos, y = logp)) +
  geom_point(aes(color = point_group, shape = MR), size = 2, alpha = 0.8) +
  scale_color_manual(
    name = "ADRD colocalization",
    values = c("Yes" = "red", "No (odd)" = "grey80", "No (even)" = "#C6DBEF"),
    breaks = c("Yes", "No (odd)"),
    labels = c("Yes", "No"),
    guide = guide_legend(order = 2)
  ) +
  scale_shape_manual(
    name = "MR",
    values = c("Significant" = 17, "Not Significant" = 16),
    labels = c("Not Significant" = "Not significant"),
    guide = guide_legend(order = 1)
  ) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_hline(
    yintercept = c(-log10(adj_cutoff), log10(adj_cutoff)),
    linetype = "dashed", color = "red"
  ) +
  geom_text_repel(
    data = lab_df,
    aes(label = gene_name),
    size = 4,
    color = "steelblue",
    box.padding = 0.3,
    max.overlaps = 50
  ) +
  scale_x_continuous(
    breaks = axis_df$center,
    labels = axis_df$chr
  ) +
  labs(
    x = "Chromosome",
    y = expression(paste("Adjusted glycoWAS ", -log[10](P)))
  ) +
  theme_classic(base_size = 16) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black", linewidth = 0.35),
    axis.ticks = element_line(color = "black", linewidth = 0.35),
    legend.position = "bottom"
  )

## Supplementary fig3 -- pQTL

In [ ]:
df <- twas_pqtl %>%
  mutate(sig = twas_pval < pqtl_cutoff) %>%
  group_by(molecular_id, context, gwas_study) %>%
  summarise(
    n_methods = n(),
    n_sig = sum(sig, na.rm = TRUE),
    best_pval = min(twas_pval, na.rm = TRUE),
    best_method_pval = min(twas_pval[is_selected_method], na.rm = TRUE),
    best_method_selected = any(sig & is_selected_method, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    pass = (n_sig / n_methods >= 0.5) | best_method_selected, # pass defined by two threshold, best or half methods satisfied
    final_pval = ifelse(n_sig / n_methods >= 0.5, best_pval, best_method_pval),
    # # fallback: if no selected method existed
    final_pval = ifelse(is.infinite(final_pval), best_pval, final_pval)  ) %>%
  left_join(twas_pqtl, by = c("molecular_id", "context", "gwas_study")) %>%
  filter(twas_pval == final_pval) %>%
  distinct(molecular_id, .keep_all = TRUE) %>%
  mutate(
    logp = -log10(twas_pval) * sign(twas_z),
    pos = (start + end) / 2,
    is_coloc = molecular_id %in% colocboost_pqtl_genes  # protein level: the pQTL itself is an AD CoS member
  ) %>%
  merge(gene_ref, by.x = 'molecular_id', by.y = 'gene_id') %>%
  mutate(peptide = paste0(gene_name, ": ", str_extract(context, 'gp_.+$')))


df$chr <- factor(df$chr, levels = as.character(1:22))

chr_info <- df %>%
  group_by(chr) %>%
  summarise(chr_len = max(pos)) %>%
  mutate(chr_start = lag(cumsum(chr_len), default = 0))

df <- df %>%
  left_join(chr_info, by = "chr") %>%
  mutate(cum_pos = pos + chr_start) %>%
  mutate(
    TWAS_colocaliztion = ifelse(
      is_coloc & pass, 
      'TWAS & Colocalization',
      ifelse(!is_coloc & pass, 'TWAS only', 'Not Significant')
    )
  )
twas_pqtl_sig_df = df
axis_df <- chr_info %>%
  mutate(center = chr_start + chr_len/2)

df <- mr_res_df_sig_pqtl %>% rename('molecular_id' = 'gene_name') %>% merge(df, . , all.x = T)

In [ ]:
library(ggplot2)
library(ggrepel)
library(dplyr)

# Match the main Figure 3a design
df <- df %>%
  mutate(
    MR = ifelse(!is.na(Q_pval), "Significant", "Not Significant"),
    chr_num = as.integer(as.character(chr)),
    point_group = case_when(
      is_coloc & pass ~ "Yes",   # colocalized points without GlycoWAS significance stay grey
      chr_num %% 2 == 1 ~ "No (odd)",
      TRUE ~ "No (even)"
    )
  ) %>% arrange(point_group == "Yes")

lab_df <- df %>%
  filter(pass) %>%
  group_by(gene_name) %>%
  slice_max(abs(logp), n = 1, with_ties = FALSE) %>%
  ungroup()

manual_names <- c("RAB8B", "CTSH", "LACTB")
auto_lab_df <- lab_df %>% filter(!gene_name %in% manual_names)
manual_lab_df <- lab_df %>%
  filter(gene_name %in% manual_names) %>%
  mutate(
    x_span = diff(range(df$cum_pos, na.rm = TRUE)),
    label_x = case_when(
      gene_name == "RAB8B" ~ cum_pos - 0.030 * x_span,
      gene_name == "CTSH" ~ cum_pos + 0.030 * x_span,
      TRUE ~ cum_pos
    ),
    label_y = case_when(
      gene_name == "RAB8B" ~ 2.8,
      gene_name == "CTSH" ~ 2.0,
      gene_name == "LACTB" ~ 13.5
    ),
    segment_y = if_else(gene_name == "LACTB", label_y - 0.45, label_y),
    label_hjust = case_when(
      gene_name == "RAB8B" ~ 1,
      gene_name == "CTSH" ~ 0,
      TRUE ~ 0.5
    )
  )

spp_f3a_pqtl <- ggplot(df, aes(x = cum_pos, y = logp)) +
  geom_point(aes(color = point_group, shape = MR), size = 2, alpha = 0.8) +
  scale_color_manual(
    name = "ADRD colocalization",
    values = c("Yes" = "red", "No (odd)" = "grey80", "No (even)" = "#C6DBEF"),
    breaks = c("Yes", "No (odd)"),
    labels = c("Yes", "No"),
    guide = guide_legend(order = 2)
  ) +
  scale_shape_manual(
    name = "MR",
    values = c("Significant" = 17, "Not Significant" = 16),
    labels = c("Not Significant" = "Not significant"),
    guide = guide_legend(order = 1)
  ) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_hline(
    yintercept = c(-log10(pqtl_cutoff), log10(pqtl_cutoff)),
    linetype = "dashed", color = "red"
  ) +
  geom_text_repel(
    data = auto_lab_df,
    aes(label = gene_name),
    size = 4,
    color = "steelblue",
    box.padding = 0.3,
    max.overlaps = 50
  ) +
  geom_segment(
    data = manual_lab_df,
    aes(x = cum_pos, y = logp, xend = label_x, yend = segment_y),
    inherit.aes = FALSE,
    color = "steelblue",
    linewidth = 0.35
  ) +
  geom_text(
    data = manual_lab_df,
    aes(x = label_x, y = label_y, label = gene_name, hjust = label_hjust),
    inherit.aes = FALSE,
    size = 4,
    color = "steelblue"
  ) +
  scale_x_continuous(
    breaks = axis_df$center,
    labels = axis_df$chr
  ) +
  labs(
    x = "Chromosome",
    y = expression(paste("PWAS ", -log[10](P)))
  ) +
  theme_classic(base_size = 16) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black", linewidth = 0.35),
    axis.ticks = element_line(color = "black", linewidth = 0.35),
    legend.position = "bottom"
  )

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 10)
library(patchwork)
library(cowplot)
top_row <- spp_f3a_adj
bottom_row <- spp_f3a_pqtl
combined_fig <- (top_row / bottom_row) +
  plot_layout(heights = c(1, 1), guides = "collect") +
  plot_annotation(tag_levels = "a", tag_prefix = "", tag_suffix = "", tag_sep = "")

combined_fig <- combined_fig & theme(
  plot.tag = element_text(face = "bold", size = 22),
  plot.tag.position = c(0, 1),
  legend.position = "bottom"
)

combined_fig

ggsave("Supplementary_Figure3.pdf", combined_fig, width = 16, height = 10, dpi = 300)